# 🌲 From-Scratch Decision Tree Classifier (NumPy)

This notebook evaluates our custom from-scratch `DecisionTree` implementation (CART algorithm) on Human Activity Recognition features and compares it directly with standard scikit-learn's baseline model.

In [ ]:
import os
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier as SklearnDecisionTree
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Ensure project root is in path for imports
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.models.decision_tree import DecisionTree as ScratchDecisionTree
from src.evaluation.metrics import accuracy, precision, recall, classification_report
from src.utils.helpers import load_config, set_seed
from scripts.train import extract_statistical_features

config = load_config("config/config.yaml")
seed = config["training"]["random_seed"]
set_seed(seed)

sns.set_theme(style="whitegrid")

## 📂 Dataset Preparation

If the raw UCI HAR Dataset is missing, we automatically fall back to generating synthetic accelerometer windows representing 6 activities.

In [ ]:
has_real_data = False
try:
    from src.data.preprocessing import load_and_split_dataset
    X_train, X_test, y_train, y_test = load_and_split_dataset(config)
    X_all = np.concatenate([X_train, X_test])
    y_all = np.concatenate([y_train, y_test])
    has_real_data = True
    print(f"✅ Successfully loaded real dataset: X shape = {X_all.shape}, y shape = {y_all.shape}")
except Exception:
    print("⚠️ Dataset not found. Generating synthetic accelerometer window dataset...")
    np.random.seed(seed)
    n_samples = 200
    seq_len = 500
    n_channels = 3
    
    X_all = np.random.randn(n_samples, seq_len, n_channels)
    y_all = np.random.choice([1, 2, 3, 4, 5, 6], size=n_samples)
    for i in range(n_samples):
        label = y_all[i]
        if label <= 3:
            X_all[i, :, 0] += np.sin(np.linspace(0, 10 * label, seq_len))
        else:
            X_all[i, :, 1] += 0.5 * label
            
    print(f"✅ Generated synthetic dataset: X shape = {X_all.shape}, y shape = {y_all.shape}")

## 📐 Feature Extraction & Splitting

We extract handcrafted statistical features (24 features) and scale them.

In [ ]:
X_features = extract_statistical_features(X_all)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_features, y_all, test_size=0.3, random_state=seed, stratify=y_all
)

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train set shape: {X_train_scaled.shape}")
print(f"Test set shape : {X_test_scaled.shape}")

## 🌲 Custom Decision Tree (From Scratch) vs Scikit-learn

We train both models and evaluate accuracy, training time, and detailed classification reports.

In [ ]:
# 1. Custom Decision Tree
tree_scratch = ScratchDecisionTree(max_depth=5, criterion="entropy")
start_t = time.perf_counter()
tree_scratch.fit(X_train_scaled, y_train)
scratch_train_time = time.perf_counter() - start_t

start_t = time.perf_counter()
y_pred_scratch = tree_scratch.predict(X_test_scaled)
scratch_pred_time = time.perf_counter() - start_t
scratch_acc = accuracy(y_test, y_pred_scratch)

print("=== custom From-Scratch Decision Tree ===")
print(f"Training Time   : {scratch_train_time:.4f}s")
print(f"Prediction Time : {scratch_pred_time:.4f}s")
print(f"Accuracy        : {scratch_acc:.4f} ({scratch_acc*100:.1f}%)")
print(f"Depth           : {tree_scratch.get_depth()}")
print(f"Leaves          : {tree_scratch.get_n_leaves()}")

# 2. Scikit-learn Baseline
tree_sklearn = SklearnDecisionTree(max_depth=5, criterion="entropy", random_state=seed)
start_t = time.perf_counter()
tree_sklearn.fit(X_train_scaled, y_train)
sklearn_train_time = time.perf_counter() - start_t

start_t = time.perf_counter()
y_pred_sklearn = tree_sklearn.predict(X_test_scaled)
sklearn_pred_time = time.perf_counter() - start_t
sklearn_acc = accuracy(y_test, y_pred_sklearn)

print("\n=== Scikit-learn Decision Tree Baseline ===")
print(f"Training Time   : {sklearn_train_time:.4f}s")
print(f"Prediction Time : {sklearn_pred_time:.4f}s")
print(f"Accuracy        : {sklearn_acc:.4f} ({sklearn_acc*100:.1f}%)")

## 📊 Classification Report (From Scratch)

We use our custom formatted classification report function (which calculates precision, recall, and f1 from scratch using NumPy) to print results.

In [ ]:
activity_names = list(config.get("activities", {}).values())
if not activity_names:
    activity_names = [f"Class {i}" for i in sorted(np.unique(y_test))]

print(classification_report(y_test, y_pred_scratch, target_names=activity_names))